# DAKGEA - Simple Experiment

In this notebook we'll run a simple baseline experiment:
- Dataset: D_W_15K_V1 (DBpedia-Wikidata)
- Reduction: 30% of entities
- No augmentation (baseline only)
- Model: BERT-INT

## Setup

In [ ]:
import sys
from pathlib import Path
import yaml

# Setup path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Create Configuration

A DAKGEA configuration defines:
- **name**: Experiment name
- **dataset**: Which dataset to use
- **reduction**: How to reduce the data
- **model**: Which EA model to use
- **seed**: For reproducibility

In [ ]:
# Define configuration
config = {
    "experiment": {
        "name": "tutorial_baseline",
        
        # Dataset configuration
        "dataset": {
            "name": "openea/D_W_15K_V1",  # DBpedia-Wikidata 15K entities
            "writer": "bert_int"          # How to write processed data
        },
        
        # Reduction: use only 30% of entities
        "reduction": {
            "method": "random_entities",   # Random selection
            "ratio": 0.3,                  # 30% of dataset
            "writer": "bert_int",
            "eval": True,                  # Evaluate baseline
            "save_dataset": False,         # Don't save reduced dataset
            "save_model": False            # Don't save models
        },
        
        # Model configuration
        "model": "bert_int",               # Use BERT-INT
        "seed": 42,                        # Random seed
        "clear": True,                     # Clear intermediate cache
        "overwrite_existing": False        # Don't overwrite if exists
    }
}

# Display config
print(yaml.dump(config, default_flow_style=False))

## 2. Save Configuration

In [ ]:
# Save configuration to file
config_file = PROJECT_ROOT / "config" / "experiments" / "tutorial_baseline.yaml"

with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to: {config_file}")

## 3. Run Experiment

### Option A: From command line

```bash
# In terminal
bash scripts/run_experiment.sh config/experiments/tutorial_baseline.yaml
```

### Option B: From Python (in this notebook)

In [ ]:
from experiments.runner.runner import ExperimentRunner
from src.config.loader import load_yaml

# Load configuration
config_data = load_yaml(config_file)

# Create runner
runner = ExperimentRunner(
    payload=config_data,
    cli_overwrite=False,
    default_overwrite=False
)

print(f"Experiment: {runner.name}")
print(f"Workspace: {runner.base_workspace}")
print(f"Models to train: {runner.models}")

In [ ]:
# RUN EXPERIMENT
# ⚠️ This may take a few minutes!
# ⚠️ Requires GPU for BERT-INT

print("Starting experiment...")
print("This may take 5-10 minutes depending on your hardware.\n")

try:
    runner.run()
    print("\n✓ Experiment completed successfully!")
except Exception as e:
    print(f"\n✗ Experiment failed: {e}")
    import traceback
    traceback.print_exc()

## 4. Analyze Results

In [ ]:
import json

# Find results file
results_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "reduction" / "results.json"

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    
    print("=" * 60)
    print("BASELINE RESULTS (30% reduction)")
    print("=" * 60)
    
    for model, metrics in results.items():
        print(f"\nModel: {model}")
        print("-" * 40)
        print(f"  Hits@1:     {metrics.get('hits@1', 'N/A'):.4f}")
        print(f"  Hits@5:     {metrics.get('hits@5', 'N/A'):.4f}")
        print(f"  Hits@10:    {metrics.get('hits@10', 'N/A'):.4f}")
        print(f"  MRR:        {metrics.get('mrr', 'N/A'):.4f}")
        print(f"  Precision:  {metrics.get('precision', 'N/A'):.4f}")
        print(f"  Recall:     {metrics.get('recall', 'N/A'):.4f}")
        print(f"  F-measure:  {metrics.get('f-measure', 'N/A'):.4f}")
else:
    print(f"✗ Results file not found: {results_file}")

## 5. Results Structure

Results are saved in:

```
results/tutorial_baseline/
├── metadata.json              # Experiment metadata
└── reduction_030/             # Ratio 0.3 (30%)
    └── D_W_15K_V1/            # Dataset
        ├── artifacts/         # Cache and intermediate files
        └── reduction/         # Reduction stage
            ├── results.json   # Results
            ├── summary.json   # Summary
            └── logs/          # Execution logs
```

## Metrics Explained

- **Hits@k**: % of times the correct entity is in the top-k results
  - Hits@1: Correct entity is the first result
  - Hits@5: Correct entity is in the top 5
  - Hits@10: Correct entity is in the top 10

- **MRR** (Mean Reciprocal Rank): Average of the reciprocal rank of correct entity
  - Higher values = better (max 1.0)

- **Precision/Recall/F-measure**: Standard classification metrics
  - Precision: % of correct alignments among predicted ones
  - Recall: % of alignments found vs total
  - F-measure: Harmonic mean of precision and recall

## Next Steps

In the next notebook we'll see how to:
- Add data augmentation using PLM
- Compare baseline vs augmented
- Analyze improvement

➡️ Continue with: **03_data_augmentation.ipynb**